# שיעור 23 — KNN: מרחק, שכנים והצבעת רוב
יחידה 3 | 2026–2027 | חלק א׳ מתוך שני שיעורי KNN.

ידע קודם: מאפיינים ויעד, ריבוע ושורש, טבלה ומיון, קנה מידה. התאמה שלנו לסיכום ולמחברת ההוראה של מפגש 5 במאגר; לא מחברת מקור של המרצה.
כל הנקודות בשיעור זה מומצאות להוראה. אין מאגר אמיתי או CSV חיצוני, ואין ציון דיוק על Test. שמרו עותק אישי.

## למה לומדים את זה עכשיו?
המלצה על סוג פעילות לפי פעילויות של תלמידים דומים: במה מודדים דמיון?
**בתהליך הפרויקט:** מודל סיווג.
**מסגרת פעילה:** 3 שעות לימוד — 135 דקות, כולל דוגמה ותרגול עצמאי. המורה בוחר את משימות הליבה שבמדריך; שאר הפעילויות לחזרה או לעיון, ללא תוספת שעות.

## פתיחה — איך מסווגים תצפית שאין לה תווית ידועה?
לדודו יש דוגמאות מסומנות משתי מחלקות ותצפית חדשה. הרעיון של KNN: למצוא K דוגמאות אימון קרובות לפי המאפיינים ולסווג לפי הצבעתן. בדוגמה זו כל שכן מצביע קול אחד, ללא שקלול מרחק.

X הם שני מאפיינים מספריים בעלי קנה מידה בר־השוואה שנבחר מראש בדוגמת ההוראה. y היא המחלקה, שאינה רכיב בחישוב המרחק. מזהה הדוגמה אינו מאפיין.

## 1. מרחק אוקלידי — פיתגורס
לשתי תצפיות (x1,x2) ו־(u1,u2):

מרחק = שורש של [(x1−u1)²+(x2−u2)²].

לדוגמה (1,2) ו־(4,6): הפרשים 3 ו־4, ריבועים 9 ו־16, סכום 25 ושורש 5. המרחק אינו הפרש בין שמות המחלקות ואינו סכום הפרשים שיכולים להתבטל.

In [ ]:
import math
distance_example = math.sqrt((4 - 1) ** 2 + (6 - 2) ** 2)
print("מרחק בדוגמה:", distance_example)

## 2. נקודה חדשה וחמש דוגמאות אימון
התצפית החדשה היא (3,3), ותוויתה האמיתית אינה ידועה לנו.

| מזהה | f1 | f2 | מחלקה |
|---|---|---|---|
| A | 3 | 2 | Red |
| B | 5 | 3 | Blue |
| C | 5 | 5 | Blue |
| D | 7 | 3 | Red |
| E | 3 | 8 | Red |

לפני קוד: חשבו לפחות שני מרחקים, וציינו איזה מידע אינו נכנס לנוסחה.

In [ ]:
import pandas as pd
training_df = pd.DataFrame({
    "id": ["A", "B", "C", "D", "E"],
    "f1": [3, 5, 5, 7, 3],
    "f2": [2, 3, 5, 3, 8],
    "label": ["Red", "Blue", "Blue", "Red", "Red"]
})
query_f1 = 3
query_f2 = 3

## 3. רואים את התצפיות — תבנית ציור מוכנה
הכוכב הוא הקלט החדש, לא תווית אמיתית. הגרף עוזר להמחשה; את הסיווג ננמק בטבלת מרחקים.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 5))
sns.scatterplot(data=training_df, x="f1", y="f2", hue="label", palette={"Red": "firebrick", "Blue": "royalblue"}, s=80)
plt.scatter([query_f1], [query_f2], marker="*", s=220, color="black", label="New observation")
plt.title("Teaching data for nearest-neighbor classification")
plt.xlabel("Feature 1 (teaching scale)")
plt.ylabel("Feature 2 (teaching scale)")
plt.axis("equal")
plt.legend()
plt.tight_layout()
plt.show()

## 4. טבלת מרחקים
נשמור טבלת עבודה נפרדת. בחישוב על עמודה, אותה פעולת חיסור/ריבוע חלה על כל ערך בעמודה. חזקה 0.5 היא שורש. זו דרך טבלאית לחישוב, לא דרישה לממש אלגוריתם מלא היום.

In [ ]:
distance_df = training_df.copy()
distance_df["distance"] = (
    (distance_df["f1"] - query_f1) ** 2 +
    (distance_df["f2"] - query_f2) ** 2
) ** 0.5
ranked_df = distance_df.sort_values("distance")
display(ranked_df)

מרחקים: A=1, B=2, C=√8≈2.83, D=4, E=5. לדוגמה C: שני הפרשים 2, ולכן √(4+4)=√8. ממיינים מן הקטן לגדול. אין למיין כל מאפיין בנפרד ולנתק תוויות מרשומות.

## 5. מצביעים לפי K
- K=1: השכן A הוא Red → תחזית Red.
- K=3: A,B,C; שני Blue מול Red אחד → תחזית Blue.
- K=5: שלושה Red מול שני Blue → תחזית Red.

השכן הקרוב ביותר אינו תמיד קובע כאשר כל הקולות שווים. שינוי K שינה תחזית, אבל איננו יודעים מה נכון בלי תווית אמת והערכה מתאימה.

In [ ]:
neighbors_1 = ranked_df.head(1)
neighbors_3 = ranked_df.head(3)
neighbors_5 = ranked_df.head(5)
print("תחזית/תחזיות K=1:")
print(neighbors_1["label"].mode())
print("תחזית/תחזיות K=3:")
print(neighbors_3["label"].mode())
print("תחזית/תחזיות K=5:")
print(neighbors_5["label"].mode())

mode מחזירה את כל המחלקות בשכיחות המרבית. בדוגמה כל בחירה נותנת מנצחת יחידה. אם יש כמה שכיחות — צריך כלל הכרעה מוגדר, ולא לבחור בשקט את הראשונה. K אי־זוגי מונע תיקו בספירת קולות בשתי מחלקות בלי קולות חסרים, אך אינו מבטיח זאת בריבוי מחלקות. גם שוויון מרחקים סביב השכן ה־K דורש מדיניות.

### עצירת תרגול מודרך
בלי לשנות נתונים, בחרו את שלושת השכנים והציגו רק id,label,distance. כתבו הסבר מילולי אחד להחלטה. מה יקרה ל־K=1 אם תווית השכן הקרוב הוזנה בטעות? זו שאלה על רגישות, לא הוראה לשנות מקור.

In [ ]:
# השלימו כאן

## 6. מה K משנה ומה לא הוכחנו?
K קטן יכול להיות רגיש לתצפית בודדת; K גדול יכול לטשטש מבנה מקומי ולהעדיף מחלקה נפוצה. אין כלל שיותר שכנים תמיד משפרים דיוק. K צריך להיות חיובי ולא גדול ממספר דוגמאות האימון.

בהמשך נבחר K בעזרת אימות בתוך האימון, לא לפי ציון Test סופי. ההמחשה כאן איננה ניסוי דיוק או כוונון על בדיקה.

KNN הוא למידה מבוססת דוגמאות: fit בשלב הספרייה ישמור את דוגמאות האימון ויכין חיפוש; הוא לא חייב ללמוד משקולות כמו רשת עצבית.

## תוצר לפרויקט וכרטיס יציאה
תעדו בעיית סיווג, מאפיינים ותווית, קנה מידה, נוסחת מרחק, K, שכנים והצבעה, תחזית ומגבלה. אין לקרוא לתחזית 'אמת'.

1. מדוע y אינה חלק מחישוב המרחק?
2. למה ממיינים מהקטן לגדול?
3. האם K=1 ו־K=3 חייבים להסכים?
4. למה שינוי K בדוגמה אינו מדד הצלחה?
5. מה תצטרכו לבדוק לפני יישום על נתונים אמיתיים?

הבא: KNN ב־scikit-learn על מאפיינים מספריים, לאחר חלוקה, השלמה ותקנון תקינים.

In [ ]:
print("ממדי מקור ההוראה:", training_df.shape)
print("עמודות מקור ההוראה:", training_df.columns)